In [4]:
import sqlite3
import pandas as pd

# Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create Orders table
cursor.execute('''
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    category TEXT NOT NULL,
    order_amount REAL NOT NULL,
    shipping_country TEXT NOT NULL,
    order_date DATE
);
''')

# Insert sample data
orders_data = [
    (1, 'John Doe', 'Electronics', 1200.50, 'USA', '2026-01-15'),
    (2, 'Jane Smith', 'Furniture', 450.00, 'Canada', '2026-01-18'),
    (3, 'Rahul Kumar', 'Electronics', 85.00, 'India', '2026-02-02'),
    (4, 'Anna Miller', 'Apparel', 45.25, 'USA', '2026-02-10'),
    (5, 'John Doe', 'Apparel', 120.00, 'USA', '2026-02-14'),
    (6, 'Carlos Ruiz', 'Furniture', 899.00, 'Mexico', '2026-03-01'),
    (7, 'Jane Smith', 'Electronics', 60.00, 'Canada', '2026-03-05'),
    (8, 'Rahul Kumar', 'Apparel', 35.00, 'India', '2026-03-12'),
    (9, 'Sophia Brown', 'Electronics', 750.00, 'UK', '2026-03-20'),
    (10, 'David Lee', 'Furniture', 1100.00, 'Singapore', '2026-03-25')
]

cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?)', orders_data)

conn.commit()

print("Sample 'orders' table created and populated successfully.")

Sample 'orders' table created and populated successfully.


### 1. Select all orders

In [10]:
cursor.execute("SELECT * FROM orders")
orders_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(orders_df)

,order_id,customer_name,category,order_amount,shipping_country,order_date
0,1,John Doe,Electronics,1200.50,USA,2026-01-15
1,2,Jane Smith,Furniture,450.00,Canada,2026-01-18
2,3,Rahul Kumar,Electronics,85.00,India,2026-02-02
3,4,Anna Miller,Apparel,45.25,USA,2026-02-10
4,5,John Doe,Apparel,120.00,USA,2026-02-14
5,6,Carlos Ruiz,Furniture,899.00,Mexico,2026-03-01
6,7,Jane Smith,Electronics,60.00,Canada,2026-03-05
7,8,Rahul Kumar,Apparel,35.00,India,2026-03-12
8,9,Sophia Brown,Electronics,750.00,UK,2026-03-20
9,10,David Lee,Furniture,1100.00,Singapore,2026-03-25


### 2. Orders from a specific country


In [11]:
# Specify the country you want to filter by
country_to_filter = 'USA' # @param {type:"string"}

# Construct the SQL query with a WHERE clause
sql_query = f"SELECT * FROM orders WHERE shipping_country = '{country_to_filter}'"

# Execute the query
cursor.execute(sql_query)

# Fetch the results and display them in a DataFrame
filtered_orders_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(filtered_orders_df)

,order_id,customer_name,category,order_amount,shipping_country,order_date
0,1,John Doe,Electronics,1200.50,USA,2026-01-15
1,4,Anna Miller,Apparel,45.25,USA,2026-02-10
2,5,John Doe,Apparel,120.00,USA,2026-02-14


### 3. Sort orders by amount

In [12]:
sql_query_sorted = "SELECT * FROM orders ORDER BY order_amount DESC"
cursor.execute(sql_query_sorted)

sorted_orders_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(sorted_orders_df)

,order_id,customer_name,category,order_amount,shipping_country,order_date
0,1,John Doe,Electronics,1200.50,USA,2026-01-15
1,10,David Lee,Furniture,1100.00,Singapore,2026-03-25
2,6,Carlos Ruiz,Furniture,899.00,Mexico,2026-03-01
3,9,Sophia Brown,Electronics,750.00,UK,2026-03-20
4,2,Jane Smith,Furniture,450.00,Canada,2026-01-18
5,5,John Doe,Apparel,120.00,USA,2026-02-14
6,3,Rahul Kumar,Electronics,85.00,India,2026-02-02
7,7,Jane Smith,Electronics,60.00,Canada,2026-03-05
8,4,Anna Miller,Apparel,45.25,USA,2026-02-10
9,8,Rahul Kumar,Apparel,35.00,India,2026-03-12


### 4. Count orders by category

In [13]:
sql_query_category_count = "SELECT category, COUNT(order_id) AS order_count FROM orders GROUP BY category ORDER BY order_count DESC"
cursor.execute(sql_query_category_count)

category_counts_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(category_counts_df)

,category,order_count
0,Electronics,4
1,Furniture,3
2,Apparel,3


### 5. Average order amount per category

In [14]:
sql_query_avg_amount = "SELECT category, AVG(order_amount) AS average_order_amount FROM orders GROUP BY category ORDER BY average_order_amount DESC"
cursor.execute(sql_query_avg_amount)

average_amounts_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(average_amounts_df)

,category,average_order_amount
0,Furniture,816.333333
1,Electronics,523.875000
2,Apparel,66.750000


### 6. High-value orders

In [15]:
# Define the threshold for high-value orders
high_value_threshold = 500.0 # @param {type:"number"}

# Construct the SQL query to select high-value orders
sql_query_high_value = f"SELECT * FROM orders WHERE order_amount > {high_value_threshold} ORDER BY order_amount DESC"
cursor.execute(sql_query_high_value)

# Fetch the results and display them in a DataFrame
high_value_orders_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(high_value_orders_df)

,order_id,customer_name,category,order_amount,shipping_country,order_date
0,1,John Doe,Electronics,1200.5,USA,2026-01-15
1,10,David Lee,Furniture,1100.0,Singapore,2026-03-25
2,6,Carlos Ruiz,Furniture,899.0,Mexico,2026-03-01
3,9,Sophia Brown,Electronics,750.0,UK,2026-03-20


### 7. Orders after a specific date

In [16]:
# Define the specific date to filter by
filter_date = '2026-03-01' # @param {type:"string"}

# Construct the SQL query to select orders after the specified date
sql_query_after_date = f"SELECT * FROM orders WHERE order_date > '{filter_date}' ORDER BY order_date ASC"
cursor.execute(sql_query_after_date)

# Fetch the results and display them in a DataFrame
orders_after_date_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(orders_after_date_df)

,order_id,customer_name,category,order_amount,shipping_country,order_date
0,7,Jane Smith,Electronics,60.0,Canada,2026-03-05
1,8,Rahul Kumar,Apparel,35.0,India,2026-03-12
2,9,Sophia Brown,Electronics,750.0,UK,2026-03-20
3,10,David Lee,Furniture,1100.0,Singapore,2026-03-25


### 8. Maximum order amount by category

In [2]:
import sqlite3
import pandas as pd

# This section is re-included to ensure the database connection, table, and data
# are available in case the initial setup cell was not run or its state was lost.
# For an in-memory database, this will create a new database with the sample data.
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    category TEXT NOT NULL,
    order_amount REAL NOT NULL,
    shipping_country TEXT NOT NULL,
    order_date DATE
);
''')

orders_data = [
    (1, 'John Doe', 'Electronics', 1200.50, 'USA', '2026-01-15'),
    (2, 'Jane Smith', 'Furniture', 450.00, 'Canada', '2026-01-18'),
    (3, 'Rahul Kumar', 'Electronics', 85.00, 'India', '2026-02-02'),
    (4, 'Anna Miller', 'Apparel', 45.25, 'USA', '2026-02-10'),
    (5, 'John Doe', 'Apparel', 120.00, 'USA', '2026-02-14'),
    (6, 'Carlos Ruiz', 'Furniture', 899.00, 'Mexico', '2026-03-01'),
    (7, 'Jane Smith', 'Electronics', 60.00, 'Canada', '2026-03-05'),
    (8, 'Rahul Kumar', 'Apparel', 35.00, 'India', '2026-03-12'),
    (9, 'Sophia Brown', 'Electronics', 750.00, 'UK', '2026-03-20'),
    (10, 'David Lee', 'Furniture', 1100.00, 'Singapore', '2026-03-25')
]

cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?)', orders_data)
conn.commit()

sql_query_max_amount = "SELECT category, MAX(order_amount) AS maximum_order_amount FROM orders GROUP BY category ORDER BY maximum_order_amount DESC"
cursor.execute(sql_query_max_amount)

max_amounts_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(max_amounts_df)

,category,maximum_order_amount
0,Electronics,1200.5
1,Furniture,1100.0
2,Apparel,120.0


### 9. Customers whose name starts with J

In [3]:
sql_query_customer_name_j = "SELECT * FROM orders WHERE customer_name LIKE 'J%'"
cursor.execute(sql_query_customer_name_j)

customer_j_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(customer_j_df)

,order_id,customer_name,category,order_amount,shipping_country,order_date
0,1,John Doe,Electronics,1200.5,USA,2026-01-15
1,2,Jane Smith,Furniture,450.0,Canada,2026-01-18
2,5,John Doe,Apparel,120.0,USA,2026-02-14
3,7,Jane Smith,Electronics,60.0,Canada,2026-03-05


### 10. Orders within a specific amount range

In [4]:
# Define the minimum and maximum order amounts to filter by
min_order_amount = 100.0 # @param {type:"number"}
max_order_amount = 500.0 # @param {type:"number"}

# Construct the SQL query to select orders within the specified range
sql_query_amount_range = f"SELECT * FROM orders WHERE order_amount BETWEEN {min_order_amount} AND {max_order_amount} ORDER BY order_amount DESC"
cursor.execute(sql_query_amount_range)

# Fetch the results and display them in a DataFrame
orders_in_range_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(orders_in_range_df)

,order_id,customer_name,category,order_amount,shipping_country,order_date
0,2,Jane Smith,Furniture,450.0,Canada,2026-01-18
1,5,John Doe,Apparel,120.0,USA,2026-02-14


### 11. Total sales by country

In [5]:
sql_query_total_sales_country = "SELECT shipping_country, SUM(order_amount) AS total_sales FROM orders GROUP BY shipping_country ORDER BY total_sales DESC"
cursor.execute(sql_query_total_sales_country)

total_sales_country_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(total_sales_country_df)

,shipping_country,total_sales
0,USA,1365.75
1,Singapore,1100.00
2,Mexico,899.00
3,UK,750.00
4,Canada,510.00
5,India,120.00


### 12. Customers with more than one order

### 13. Highest order overall

In [7]:
sql_query_highest_order = "SELECT MAX(order_amount) AS highest_order_overall FROM orders"
cursor.execute(sql_query_highest_order)

highest_order_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(highest_order_df)

,highest_order_overall
0,1200.5


In [6]:
sql_query_multiple_orders = """SELECT customer_name, COUNT(order_id) AS total_orders FROM orders GROUP BY customer_name HAVING COUNT(order_id) > 1 ORDER BY total_orders DESC"""
cursor.execute(sql_query_multiple_orders)

multiple_orders_df = pd.DataFrame(cursor.fetchall(), columns=[description[0] for description in cursor.description])
display(multiple_orders_df)

,customer_name,total_orders
0,Rahul Kumar,2
1,John Doe,2
2,Jane Smith,2
